In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

laptop=pd.read_csv("laptop_prices.csv")
# print(housing)

# different catagories of prices
laptop['Price_cat']=pd.cut(
    laptop['Price_euros'],
    bins=[0,500,1000,1500,2000,np.inf],
    labels=[1,2,3,4,5])

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index,test_index in split.split(laptop,laptop['Price_cat']):
    strat_train_set= laptop.loc[train_index].drop("Price_cat", axis=1)
    strat_test_set = laptop.loc[test_index].drop("Price_cat", axis=1)


laptop=strat_train_set.copy()

laptop_labels=laptop['Price_euros'].copy()
laptop=laptop.drop('Price_euros',axis=1)


In [2]:
#separate num and cat

num_attribs = laptop.drop(columns=['Company','Product','TypeName','OS','Screen','Touchscreen'
                                            ,'IPSpanel','RetinaDisplay','CPU_company','CPU_model','PrimaryStorageType'
                                            ,'SecondaryStorageType','GPU_company','GPU_model'], axis=1).columns.tolist()

cat_attribs =['Company','Product','TypeName','OS','Screen','Touchscreen'
               ,'IPSpanel','RetinaDisplay','CPU_company','CPU_model','PrimaryStorageType'
               ,'SecondaryStorageType','GPU_company','GPU_model']

In [3]:
# 5. Pipelines

# Numerical pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])


# Categorical pipeline
cat_pipeline = Pipeline([
    ("c_imputer",SimpleImputer(strategy="most_frequent"))
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

#full pipeline
full_pipeline = ColumnTransformer([
    ("num",num_pipeline,num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])

# 6. Transform the data
laptop_prepared = full_pipeline.fit_transform(laptop)
# print(laptop_prepared.shape)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score

# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(laptop_prepared, laptop_labels)

# Decision Tree
tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(laptop_prepared, laptop_labels)

# Random Forest
forest_reg = RandomForestRegressor(random_state=42)
forest_reg.fit(laptop_prepared, laptop_labels)

ridge_reg = Ridge(alpha=1.0)

# Predict using training data
lin_preds = lin_reg.predict(laptop_prepared)
tree_preds = tree_reg.predict(laptop_prepared)
forest_preds = forest_reg.predict(laptop_prepared)



# Calculate RMSE
# lin_rmse = root_mean_squared_error(housing_labels, lin_preds)
lin_rmses =-cross_val_score(
    lin_reg,
    laptop_prepared,
    laptop_labels,
    scoring="neg_root_mean_squared_error",
    cv=10
)

tree_rmses =-cross_val_score(
    tree_reg,
    laptop_prepared,
    laptop_labels,
    scoring="neg_root_mean_squared_error",
    cv=10
)

random_rmses =-cross_val_score(
    forest_reg,
    laptop_prepared,
    laptop_labels,
    scoring="neg_root_mean_squared_error",
    cv=10
)

ridge_rmses = -cross_val_score(
    ridge_reg,
    laptop_prepared,
    laptop_labels,
    scoring="neg_root_mean_squared_error",
    cv=10
)

# forest_rmse = root_mean_squared_error(housing_labels, forest_preds)

# print("Decision Tree CV RMSEs:", tree_rmses)
print("\nCross-Validation Performance (Decision Tree):")
print(pd.Series(tree_rmses).describe())

# print("linear CV RMSEs:", lin_rmses)
print("\nCross-Validation Performance (linear):")
print(pd.Series(lin_rmses).describe())

# print("linear CV RMSEs:", random_rmses)
print("\nCross-Validation Performance (random forest):")
print(pd.Series(random_rmses).describe())

print("\nCross-Validation Performance (Ridge Regression):")
print(pd.Series(ridge_rmses).describe())
